# Fraud Detection — Exploratory Data Analysis

This notebook explores the three datasets used in the project.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid')
%matplotlib inline

BASE = os.path.join(os.getcwd(), '..')
txn_df = pd.read_csv(os.path.join(BASE, 'data/raw/transactions.csv'))
cus_df = pd.read_csv(os.path.join(BASE, 'data/raw/customers.csv'))
cc_df  = pd.read_csv(os.path.join(BASE, 'data/raw/creditcard.csv'))

print('Transactions:', txn_df.shape)
print('Customers   :', cus_df.shape)
print('Credit Card :', cc_df.shape)

## 1. Fraud Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PaySim fraud
txn_df['isFraud'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#22c55e','#ef4444'], edgecolor='black')
axes[0].set_title('PaySim: Fraud vs Legit')
axes[0].set_xticklabels(['Legit','Fraud'], rotation=0)
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():,}', (p.get_x()+p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=10)

# Credit card fraud
cc_df['Class'].value_counts().plot(kind='bar', ax=axes[1],
    color=['#22c55e','#ef4444'], edgecolor='black')
axes[1].set_title('Credit Card: Fraud vs Legit')
axes[1].set_xticklabels(['Legit','Fraud'], rotation=0)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():,}', (p.get_x()+p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=10)

plt.suptitle('Class Imbalance in Both Datasets', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'PaySim fraud rate    : {txn_df["isFraud"].mean():.4%}')
print(f'Credit Card fraud rate: {cc_df["Class"].mean():.4%}')

## 2. Transaction Amount Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution by fraud label (PaySim)
txn_df[txn_df['isFraud']==0]['amount'].clip(upper=1e6).plot(
    kind='hist', bins=60, alpha=0.6, label='Legit', ax=axes[0], color='steelblue')
txn_df[txn_df['isFraud']==1]['amount'].clip(upper=1e6).plot(
    kind='hist', bins=60, alpha=0.7, label='Fraud', ax=axes[0], color='crimson')
axes[0].set_title('PaySim: Amount Distribution'); axes[0].legend()
axes[0].set_xlabel('Amount')

# Amount distribution (Credit Card)
cc_df[cc_df['Class']==0]['Amount'].clip(upper=500).plot(
    kind='hist', bins=60, alpha=0.6, label='Legit', ax=axes[1], color='steelblue')
cc_df[cc_df['Class']==1]['Amount'].clip(upper=500).plot(
    kind='hist', bins=60, alpha=0.7, label='Fraud', ax=axes[1], color='crimson')
axes[1].set_title('Credit Card: Amount Distribution'); axes[1].legend()
axes[1].set_xlabel('Amount')

plt.tight_layout()
plt.show()

## 3. Transaction Type vs Fraud (PaySim)

In [ ]:
fraud_by_type = txn_df.groupby('type')['isFraud'].agg(['sum','count'])
fraud_by_type['rate'] = fraud_by_type['sum'] / fraud_by_type['count']
print(fraud_by_type.sort_values('rate', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fraud_by_type['sum'].sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], color='crimson', edgecolor='black')
axes[0].set_title('Fraud Count by Type'); axes[0].set_ylabel('Fraud Transactions')
axes[0].tick_params(axis='x', rotation=30)

fraud_by_type['rate'].sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color='orange', edgecolor='black')
axes[1].set_title('Fraud Rate by Type'); axes[1].set_ylabel('Fraud Rate')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Credit Card Correlation Heatmap (PCA Features)

In [ ]:
# Top 15 features most correlated with fraud
corr = cc_df.corr()['Class'].drop('Class').abs().sort_values(ascending=False).head(15)
print('Top 15 features correlated with fraud:')
print(corr)

plt.figure(figsize=(8, 5))
corr.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Correlation with Fraud (abs)')
plt.xlabel('|Correlation|')
plt.tight_layout()
plt.show()

## 5. Customer Dataset EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

cus_df['CreditScore'].plot(kind='hist', bins=40, ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Credit Score Distribution')

cus_df['Age'].plot(kind='hist', bins=40, ax=axes[0,1], color='green')
axes[0,1].set_title('Age Distribution')

cus_df['Balance'].plot(kind='hist', bins=40, ax=axes[0,2], color='orange')
axes[0,2].set_title('Balance Distribution')

cus_df['Geography'].value_counts().plot(kind='bar', ax=axes[1,0], color='purple', edgecolor='black')
axes[1,0].set_title('Geography Distribution'); axes[1,0].tick_params(axis='x', rotation=0)

cus_df['Exited'].value_counts().plot(kind='pie', ax=axes[1,1],
    autopct='%1.1f%%', labels=['Active','Churned'], colors=['#22c55e','#ef4444'])
axes[1,1].set_title('Customer Churn'); axes[1,1].set_ylabel('')

cus_df.groupby('Geography')['Balance'].mean().plot(
    kind='bar', ax=axes[1,2], color='teal', edgecolor='black')
axes[1,2].set_title('Avg Balance by Geography'); axes[1,2].tick_params(axis='x', rotation=0)

plt.suptitle('Customer Dataset Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Hourly Fraud Pattern (PaySim)

In [ ]:
txn_df['hour'] = txn_df['step'] % 24
hourly = txn_df.groupby('hour')['isFraud'].agg(['sum','count'])
hourly['rate'] = hourly['sum'] / hourly['count']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
hourly['sum'].plot(kind='bar', ax=axes[0], color='crimson', edgecolor='black')
axes[0].set_title('Fraud Count by Hour of Day')
axes[0].set_xlabel('Hour'); axes[0].set_ylabel('Fraud Count')

hourly['rate'].plot(kind='line', ax=axes[1], color='orange', marker='o', linewidth=2)
axes[1].set_title('Fraud Rate by Hour of Day')
axes[1].set_xlabel('Hour'); axes[1].set_ylabel('Fraud Rate')

plt.tight_layout()
plt.show()